In [1]:
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv
import json

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
name = 'logitech_c930e_m' # logitech_c930e_m / obj_000001

target_mesh_file = f'./models_target/models_cad/{name}.obj'

In [3]:
result = gf.compute_best_patch_pairs(
    mesh_path=target_mesh_file,
    mesh_max_triangles = 1000,
    angle_deg=30,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
    coplanar_tol=1,            # 공면성 허용 오차 (↑면 패치 수 ↓)
    min_opening=5.0,          # 그리퍼 최소 개구(mm)
    max_opening=140.0,         # 그리퍼 최대 개구(mm)
    angle_tolerance_deg=10,    # 패치 페어 정반대 threshold
    top_k=100                  # 상위 패치 페어 후보 개수
)

# reports = gf.check_gripper_feasibility_with_yaw(result)
reports = gf.check_gripper_feasibility_faces_with_yaw(result)

In [4]:
fig = gfv.visualize_merged_patches_plotly(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_normal.html", include_plotlyjs="cdn", full_html=True)

In [5]:
fig = gfv.visualize_pairs_centroid_lines(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs.html", include_plotlyjs="cdn", full_html=True)

In [6]:
fig = gfv.visualize_feasible_pairs_pads(result, reports, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs_feasible.html", include_plotlyjs="cdn", full_html=True)

In [7]:
fig = gfv.visualize_feasible_pairs_pads_gripper(result, reports, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs_grip_feasible.html", include_plotlyjs="cdn", full_html=True)

In [17]:
fig = gfv.visualize_patch_pair_faces(result, reports[0], show=True)

In [ ]:
report = reports[0]


{'pair_index': 0,
 'patch_i': 77,
 'patch_j': 79,
 'face_i': np.int64(541),
 'face_j': np.int64(578),
 'feasible': True,
 'feasible_yaw': 0,
 'moment': 0.3789788247432559}

In [13]:
pairind = 2
p_i = result['patches'][reports[pairind]['patch_i']]
p_j = result['patches'][reports[pairind]['patch_j']]
yaw_deg = reports[pairind]['feasible_yaw']

H_OG, stroke = gf.build_gripper_pose_obj(p_i, p_j, yaw_deg)
H_OG[:3, :3], H_OG[:3, 3] # O좌표계 기준 G좌표계 원점 위치

(array([[ 3.55844814e-07, -1.00000000e+00, -0.00000000e+00],
        [ 9.92546138e-01,  3.53192396e-07, -1.21869453e-01],
        [-1.21869453e-01, -4.33666130e-08, -9.92546138e-01]]),
 array([17.96656156, 40.41348156, 61.33706643]))

In [12]:
# SAM6D result 
with open("./RUN_result/output/detection_pem_20250918_134200_logitech_c930e_m.json", "r") as f:
    detections = json.load(f)

# score 가장 높은 detection 선택
best_det = max(detections, key=lambda d: d["score"])
R_oc = np.array(best_det["R"])  # (3x3)
t_oc = np.array(best_det["t"]).reshape(3, 1)  # (3x1)

H_OC = gf.to44(R_oc, t_oc) # C좌표계 기준 O좌표계 원점 위치
H_OC[:3, :3], H_OC[:3, 3]

(array([[-0.18766896, -0.03160858, -0.98172379],
        [ 0.97523892, -0.12505487, -0.18240285],
        [-0.11700404, -0.99164617,  0.05429494]]),
 array([-163.90452576, -161.02955627,  617.36737061]))

In [16]:
from scipy.spatial.transform import Rotation

feasible_r = [r for r in reports if r.get('feasible')]
print(f'[GRP] feasible gripping sol.: {len(feasible_r)}')

mesh_patches = result['mesh_patches']

feasible_pairs = []
H_dict = {"EE origin": np.eye(4,4)} # EE 좌표계 원점 
for k in range(min(5, len(feasible_r))):
    report = reports[k]
    pi = result['patches'][report['patch_i']]
    pj = result['patches'][report['patch_j']]
    ni = gf.unit(np.asarray(pi["normal"], float))
    nj = gf.unit(np.asarray(pj["normal"], float))
    fi = report['face_i']
    fj = report['face_j']
    ci = mesh_patches.vertices[mesh_patches.faces[fi]].mean(axis=0)
    cj = mesh_patches.vertices[mesh_patches.faces[fj]].mean(axis=0)
    yaw_deg = report['feasible_yaw']

    H_OG, stroke = gf.build_gripper_pose_obj({"centroid": ci, "normal": ni}, {"centroid": cj, "normal": nj}, yaw_deg)
    # H_OG, stroke = gf.build_gripper_pose_obj(pi, pj, yaw_deg)
    H_EdEn, H_OdEn = gf.ee_delta_pose_des(H_OC, H_OG)
    
    r_quat = Rotation.from_matrix(H_EdEn[:3,:3]).as_quat()
    t_quat = H_EdEn[:3,3] / 1000
    res = {"pose_quat": np.concatenate([t_quat, r_quat]),
           "stroke": stroke}
    feasible_pairs.append(res)

    H_dict[f"pair {k+1}"] = H_EdEn

gfv.visualize_frames(H_dict, scale=100, H_OdEn=H_OdEn, result=result)

[GRP] feasible gripping sol.: 12


In [ ]:
# 좌표계 시각화
H_E = np.eye(4,4)

# 고정변환
H_GnEn = gf.to44(gf.Rotx(180) @ gf.Rotz(90), [0,0,-135])   # EE -> Grip  TODO: 로봇 컨트롤러 신호 받아 변환행렬 만들기
H_GnCn = gf.to44(np.eye(3), [0,48,6])           # Cam -> Grip
H_CnGn = np.linalg.inv(H_GnCn)
H_CnEn = H_GnEn @ H_CnGn                        # EE -> Cam
H_EG = np.linalg.inv(H_GnEn)

H_OdCn = H_OC
H_OdEn = H_CnEn @ H_OdCn
# H_GdOd = np.linalg.inv(H_OG)
H_GdEn = H_OdEn @ H_OG
H_EdEn = H_GdEn @ H_EG

H_dict = {
    "E": H_E, # 엔드이펙터
    "G": H_GnEn, 
    "C": H_CnEn,
    "O_d": H_OdEn, # 목표 
    "G_d": H_GdEn,
    "E_d": H_EdEn, # EE 상대 Pose
}
fig = gfv.visualize_frames(H_dict, scale=100)

In [ ]:
H_EdEn[:3, :3], H_EdEn[:3, 3]

In [ ]:
import os
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv

cads = [file for file in os.listdir('./models_target/models_cad/') if file.endswith('.obj')]

for cad in cads:
    name = cad.split('.')[0]
    target_mesh_file = f'./models_target/models_cad/{name}.obj'


    result = gf.compute_best_patch_pairs(
        mesh_path=target_mesh_file,
        mesh_max_triangles = 1000,
        angle_deg=30,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
        coplanar_tol=1,            # 공면성 허용 오차 (↑면 패치 수 ↓)
        min_opening=5.0,          # 그리퍼 최소 개구(mm)
        max_opening=140.0,         # 그리퍼 최대 개구(mm)
        angle_tolerance_deg=10,   # 패치 페어 정반대 threshold
        top_k=100                 # 상위 패치 페어 후보 개수
    )

    # reports = gf.check_gripper_feasibility_with_yaw(result)
    reports = gf.check_gripper_feasibility_faces_with_yaw(result)

    feasible_p = result['top_k']
    feasible_r = [r for r in reports if r.get('feasible')]
    print(f'{name} : patch pairs {len(feasible_p)}, feasible sol. {len(feasible_r)}')

    fig = gfv.visualize_merged_patches_plotly(result)
    fig.write_html(f"./Grasping_Result/3-patch_normal_{name}.html", include_plotlyjs="cdn", full_html=True)

    fig = gfv.visualize_pairs_centroid_lines(result)
    fig.write_html(f"./Grasping_Result/2-patch_pairs_{name}.html", include_plotlyjs="cdn", full_html=True)

    try:
        fig = gfv.visualize_feasible_pairs_pads(result, reports)
        fig.write_html(f"./Grasping_Result/1-patch_pairs_feasible_{name}.html", include_plotlyjs="cdn", full_html=True)
    except:
        pass

logitech_c930e_m : patch pairs 31, feasible sol. 13
obj_000001 : patch pairs 64, feasible sol. 8
obj_000002 : patch pairs 74, feasible sol. 17
obj_000003 : patch pairs 63, feasible sol. 8
obj_000004 : patch pairs 33, feasible sol. 10
obj_000005 : patch pairs 63, feasible sol. 9
obj_000006 : patch pairs 86, feasible sol. 11
obj_000007 : patch pairs 72, feasible sol. 7


#### Test

In [ ]:
name = 'logitech_c930e_m' # logitech_c930e_m / obj_000001
target_mesh_file = f'./models_target/models_cad/{name}.obj'

In [ ]:
mesh_path = target_mesh_file
mesh_max_triangles = 1000
angle_deg=30              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
coplanar_tol=1            # 공면성 허용 오차 (↑면 패치 수 ↓)
min_opening=5.0           # 그리퍼 최소 개구(mm)
max_opening=140.0         # 그리퍼 최대 개구(mm)
angle_tolerance_deg=10    # 패치 페어 정반대 threshold
top_k=100                 # 상위 패치 페어 후보 개수

In [ ]:
mesh_og = trimesh.load(target_mesh_file)

In [ ]:
remesh, mesh_quad = gf.load_uniform_mesh_with_open3d(mesh_path, target_triangles=mesh_max_triangles)

In [ ]:
remesh2 = gf.split_long_edges(remesh)

In [ ]:
fig = gfv.visualize_mesh_with_edges(remesh2, edge_width=3)

In [ ]:
patches = gf.extract_planar_patches(remesh2, angle_deg=angle_deg, coplanar_tol=coplanar_tol)
patches = gf.orient_patch_normals(mesh_quad, patches, inward=False)  # false: 모두 바깥쪽으로 정렬

len(patches)

In [ ]:
params = gf.PatchPairParams(
    min_opening=min_opening,
    max_opening=max_opening,
    angle_tolerance_deg=angle_tolerance_deg,
)

In [ ]:
np.linalg.norm(patches[70].centroid)

In [ ]:
patches[70].centroid

In [ ]:
cand = gf.score_patch_pair(patches[70], patches[74], params, remesh)
cand

In [ ]:
len(cands)